<a href="https://colab.research.google.com/github/deruke/aisecops-labs/blob/main/notebooks/Lab2_Transformer_Attention_Visualization.ipynb" target="_new"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Exploring Transformer Internals — Attention Visualization
## Looking Inside the Black Box

**Workshop:** Attacking, Defending & Leveraging Agentic AI  
**Authors:** Derek Banks and Dr. Brian Fehrman  
**Time:** 35 minutes  
**Platform:** Google Colab (CPU or T4)  
**Theme:** Leverage  

---

### What You Will Learn
- How attention mechanisms work inside transformers at a practical, hands-on level
- How to extract and visualize attention weights from a real production model
- How attention patterns differ between benign and malicious/suspicious text
- What "the model is looking at" when it processes security-relevant content

### Prerequisites
- Basic understanding of neural networks (weights, layers, forward pass)
- Familiarity with Python and NumPy
- Completion of (or familiarity with) the Transformer Architecture slides
- A Google account for Colab access

### Connection to Course Material
In the lecture, we covered the transformer architecture at a conceptual level —  
queries, keys, values, multi-head attention, and how these components combine  
to let models understand language. In this lab, we **open up a real model** and  
look directly at the attention weights. This is not a toy example — DistilBERT  
is a production-grade model used in real NLP systems. Understanding what the  
model attends to is the foundation for both **leveraging** AI (interpretability,  
trust) and **attacking** it (adversarial examples that fool attention).

### Key References
- Vaswani et al. (2017): "Attention Is All You Need" — the foundational paper
- Sanh et al. (2019): "DistilBERT, a distilled version of BERT"
- Clark et al. (2019): "What Does BERT Look At? An Analysis of BERT's Attention"

In [ ]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
# Install required packages for attention extraction and
# visualization. This lab runs on CPU — no GPU required.
# ============================================================

!pip install -q transformers torch matplotlib seaborn numpy

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from transformers import DistilBertTokenizer, DistilBertModel
import warnings
import sys

warnings.filterwarnings('ignore')

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print(f"(CPU is fine for this lab — DistilBERT is small and fast)")
print("=" * 60)
print("All packages installed and imported successfully!")

---
## Background: The Attention Mechanism

Recall from the lecture that the **self-attention mechanism** is the core innovation
of the transformer architecture (Vaswani et al., 2017). Here is a quick refresher:

### How Attention Works

For each token in the input sequence, the model computes three vectors:
- **Query (Q):** "What am I looking for?"
- **Key (K):** "What do I contain?"
- **Value (V):** "What information do I carry?"

The attention score between token *i* and token *j* is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

After the softmax, each row sums to 1.0, creating a probability distribution over all tokens.
A value of 0.5 means "this token pays 50% of its attention to that token."

### Why This Matters for Security

When a model processes text like `"URGENT: Click here to verify your account"`,
the attention pattern reveals **which words the model links together**. If "URGENT"
strongly attends to "verify" and "account," the model has learned to associate urgency
cues with credential-related actions -- exactly the pattern a phishing classifier should detect.

### DistilBERT Architecture

| Component | Value |
|-----------|-------|
| Layers | 6 (vs. BERT's 12) |
| Attention heads per layer | 12 |
| Total attention patterns | 72 (6 x 12) |
| Hidden size | 768 |
| Parameters | 66M (vs. BERT's 110M) |

Each of the 72 attention heads learns to capture **different types of relationships**
between tokens. Some heads track syntax (subject-verb), others track semantics
(co-reference), and some learn positional patterns (attend to nearby tokens).

### What We Will Do in This Cell

Load the DistilBERT model and its tokenizer from Hugging Face. We set
`output_attentions=True` so the model returns the raw attention weight matrices
alongside its predictions. This is not the default behavior -- models normally
discard attention weights after computation to save memory.


In [ ]:
# ============================================================
# Cell 2: Load the Model
# ============================================================
# We use DistilBERT — a smaller, faster version of BERT that
# retains 97% of BERT's language understanding capability.
# Crucially, we set output_attentions=True so the model
# returns the raw attention weight matrices.
# ============================================================

print("Loading DistilBERT model and tokenizer...")
print("(This may take a moment on first run as weights are downloaded)\n")

model_name = "distilbert-base-uncased"

tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertModel.from_pretrained(model_name, output_attentions=True)
model.eval()  # Set to evaluation mode — no dropout, deterministic

# ---- Model Architecture Summary ----
num_layers = model.config.n_layers
num_heads = model.config.n_heads
hidden_size = model.config.hidden_size
head_dim = hidden_size // num_heads
vocab_size = model.config.vocab_size
max_seq_len = model.config.max_position_embeddings

print("=" * 60)
print("MODEL ARCHITECTURE")
print("=" * 60)
print(f"Model:             {model_name}")
print(f"Layers:            {num_layers}")
print(f"Attention heads:   {num_heads} per layer")
print(f"Hidden size:       {hidden_size}")
print(f"Head dimension:    {head_dim} (hidden_size / num_heads)")
print(f"Vocabulary size:   {vocab_size:,} tokens")
print(f"Max sequence len:  {max_seq_len}")
print(f"Total parameters:  {sum(p.numel() for p in model.parameters()):,}")
print("=" * 60)
print(f"\nWith output_attentions=True, each forward pass returns")
print(f"{num_layers} attention matrices, each of shape:")
print(f"  (batch_size, {num_heads} heads, seq_len, seq_len)")
print(f"\nThat's {num_layers} x {num_heads} = {num_layers * num_heads} individual attention patterns to explore!")

---
## Part 1: Tokenization Deep Dive

Before we can visualize attention, we need to understand **tokenization** --
how raw text becomes the numerical input the model actually processes.

### Why Tokenization Matters

Attention operates at the **token level**, not the word level. If you don't understand
tokenization, you can't correctly interpret attention heatmaps. A word that seems like
one unit ("phishing") might be split into multiple tokens ("phi", "##shing"), and each
sub-token gets its own attention weights.

### WordPiece Tokenization

DistilBERT uses **WordPiece** tokenization (Wu et al., 2016):
1. Start with a vocabulary of ~30,000 common subword units
2. Common words like "the", "login", "password" stay as single tokens
3. Rare words are split: "phishing" might become `["phi", "##shing"]`
4. The `##` prefix means "this is a continuation of the previous token"

### Special Tokens

Every input is wrapped with special tokens:
- **[CLS]** — added at the start; its final representation is used for classification
- **[SEP]** — added at the end; marks the boundary of the input sequence
- **[PAD]** — fills remaining space to reach a fixed sequence length

### What to Look For

In this cell, we tokenize several security-relevant examples. Notice:
- How technical terms (IP addresses, file paths) get split into many sub-tokens
- How common English words stay as single tokens
- How the total token count differs from the word count


In [ ]:
# ============================================================
# Cell 3: Tokenization Deep Dive
# ============================================================
# Show how text becomes tokens. Understanding tokenization is
# essential because attention weights correspond to TOKENS,
# not words. Some words get split into multiple subword tokens.
# ============================================================

examples = [
    "The password was compromised",
    "SQL injection attack detected from 192.168.1.100",
    "User authentication failed: invalid credentials",
    "URGENT: Your account has been compromised click here immediately",
    "The quarterly report is attached for your review"
]

print("=" * 70)
print("TOKENIZATION EXAMPLES")
print("=" * 70)

for text in examples:
    # Tokenize
    encoded = tokenizer(text, return_tensors='pt')
    token_ids = encoded['input_ids'][0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    print(f"\nOriginal text:  \"{text}\"")
    print(f"Token count:    {len(tokens)} (including [CLS] and [SEP])")
    print(f"Tokens:         {tokens}")
    print(f"Token IDs:      {token_ids}")
    print("-" * 70)

print("\n** KEY OBSERVATIONS **")
print("- [CLS] is prepended to every input (used for classification tasks)")
print("- [SEP] marks the end of the sequence")
print("- '##' prefix means this is a continuation of the previous word")
print("  (e.g., 'compromised' might become ['com', '##pro', '##mised'])")
print("- Numbers and special characters often get their own tokens")
print("- The model ONLY sees these tokens — attention maps will have")
print("  one row/column per token, not per word")

---
## Part 2: Extracting Attention Weights

Now we run the model on a sentence and extract the raw attention matrices.

### What Happens Inside the Model

When you call `model(**inputs, output_attentions=True)`, the forward pass:
1. **Embeds** each token into a 768-dimensional vector
2. **Passes** through 6 transformer layers sequentially
3. At **each layer**, 12 attention heads independently compute attention weights
4. **Returns** a tuple of 6 attention tensors (one per layer)

### Attention Tensor Shape

Each layer's attention tensor has shape:

```
[batch_size, num_heads, sequence_length, sequence_length]
```

For a single sentence with 15 tokens: `[1, 12, 15, 15]`

This means: for each of the 12 heads, we get a 15x15 matrix where cell (i, j)
tells us how much token *i* attends to token *j*.

### Key Properties
- Each row sums to **1.0** (softmax normalization)
- Values range from **0.0 to 1.0** (attention probabilities)
- The matrix is **NOT symmetric** -- token A attending to token B does not imply B attends to A

### What to Look For

In the output, note the shape of the attention tensors. We'll print the shape
of each layer's output so you can see the dimensions concretely.


In [ ]:
# ============================================================
# Cell 4: Extract Attention Weights
# ============================================================
# Run a forward pass and inspect the attention output shapes.
# This is the core operation — extracting what the model
# actually "looks at" when processing text.
# ============================================================

text = "The server detected a suspicious login attempt from an unknown IP address"

# Tokenize and run inference
inputs = tokenizer(text, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())

with torch.no_grad():  # No gradient computation needed for inference
    outputs = model(**inputs)

# The attentions are returned as a tuple of tensors, one per layer
attentions = outputs.attentions

print("=" * 60)
print("ATTENTION WEIGHT EXTRACTION")
print("=" * 60)
print(f"Input text:    \"{text}\"")
print(f"Tokens:        {tokens}")
print(f"Token count:   {len(tokens)}")
print(f"\nAttention output type: {type(attentions)}")
print(f"Number of layers:      {len(attentions)}")
print(f"\nShape per layer:")
for i, attn in enumerate(attentions):
    print(f"  Layer {i}: {attn.shape}")
    # Shape: (batch_size, num_heads, seq_len, seq_len)

print(f"\n** INTERPRETING THE SHAPE **")
print(f"Shape: (batch={attentions[0].shape[0]}, "
      f"heads={attentions[0].shape[1]}, "
      f"seq_len={attentions[0].shape[2]}, "
      f"seq_len={attentions[0].shape[3]})")
print(f"\n- batch:   We're processing 1 sentence at a time")
print(f"- heads:   {num_heads} parallel attention patterns per layer")
print(f"- seq_len: {len(tokens)} tokens in this sentence")
print(f"- The last two dimensions form a {len(tokens)}x{len(tokens)} matrix")
print(f"  where entry [i][j] = how much token i attends to token j")
print(f"\n- Each ROW sums to 1.0 (softmax normalization)")

# Verify: each row sums to 1
row_sums = attentions[0][0, 0].sum(dim=-1)
print(f"- Verification: row sums = {row_sums.numpy().round(4)}")

---
## Part 3: Visualizing a Single Attention Head

Now we create our first attention heatmap. This is where things get visual.

### How to Read an Attention Heatmap

```
             Token_A  Token_B  Token_C  Token_D
Token_A      [ 0.1     0.7     0.1     0.1  ]  ← Token_A pays 70% attention to Token_B
Token_B      [ 0.3     0.2     0.4     0.1  ]  ← Token_B pays 40% attention to Token_C
Token_C      [ 0.1     0.1     0.1     0.7  ]  ← Token_C pays 70% attention to Token_D
Token_D      [ 0.5     0.2     0.2     0.1  ]  ← Token_D pays 50% attention to Token_A
```

- **Y-axis (rows):** The token that is "looking" (the query)
- **X-axis (columns):** The token being "looked at" (the key)
- **Color intensity:** Strength of attention (brighter = stronger)
- Each row sums to 1.0

### What Different Patterns Mean

| Pattern | What It Looks Like | What It Means |
|---------|-------------------|---------------|
| **Diagonal** | Bright diagonal line | Tokens attend to themselves |
| **Vertical stripe** | One bright column | All tokens attend to one specific token (often [CLS] or [SEP]) |
| **Block** | Bright rectangular region | A group of tokens strongly attend to each other |
| **Scattered** | No clear pattern | Distributed attention across many tokens |

### What We Visualize

We pick **Layer 3, Head 0** as a starting point. This is a middle layer where
attention patterns tend to be more semantically meaningful than the very first
or very last layers. The choice of head is somewhat arbitrary -- we'll explore
all heads later in Part 9.


In [ ]:
# ============================================================
# Cell 5: Visualize a Single Attention Head
# ============================================================
# Create a detailed heatmap for one specific attention head.
# This shows the raw attention pattern — what each token
# "looks at" in the input.
# ============================================================

def visualize_attention_head(attentions, tokens, layer, head,
                             title_suffix="", figsize=(10, 8), cmap="YlOrRd"):
    """
    Visualize a single attention head as a heatmap.

    Args:
        attentions: tuple of attention tensors from model output
        tokens: list of token strings
        layer: which layer (0-indexed)
        head: which head (0-indexed)
        title_suffix: optional string appended to title
        figsize: figure dimensions
        cmap: matplotlib colormap
    """
    # Extract the attention matrix for this layer and head
    attn_matrix = attentions[layer][0, head].detach().numpy()

    fig, ax = plt.subplots(figsize=figsize)

    # Create heatmap
    im = ax.imshow(attn_matrix, cmap=cmap, aspect='auto', vmin=0, vmax=1)

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Attention Weight', fontsize=11)

    # Set tick labels to tokens
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(tokens, fontsize=9)

    # Labels
    ax.set_xlabel('Attended To (Keys)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Attending From (Queries)', fontsize=12, fontweight='bold')
    ax.set_title(f'Attention: Layer {layer}, Head {head} {title_suffix}',
                 fontsize=14, fontweight='bold')

    # Add value annotations for small sequences
    if len(tokens) <= 16:
        for i in range(len(tokens)):
            for j in range(len(tokens)):
                val = attn_matrix[i, j]
                color = 'white' if val > 0.5 else 'black'
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=7, color=color)

    plt.tight_layout()
    plt.show()

    return attn_matrix


# Visualize Layer 0, Head 0
print("Layer 0, Head 0 — the first attention pattern the model computes:")
print("(Read: row token ATTENDS TO column token)\n")
_ = visualize_attention_head(attentions, tokens, layer=0, head=0)

# Also show a deeper layer for comparison
print(f"\nLayer {num_layers-1}, Head 0 — the deepest layer's first head:")
print("(Notice how attention patterns differ from Layer 0)\n")
_ = visualize_attention_head(attentions, tokens, layer=num_layers-1, head=0)

---
## Part 4: Security-Relevant Text Comparison

This is the core of the lab. We compare attention patterns between **benign** and
**suspicious/malicious** text to see if the model processes them differently.

### Why This Matters

NLP-based security tools (phishing detectors, malware name classifiers, log anomaly
detectors) all use transformer models internally. Understanding how these models
distribute attention across tokens helps us:

1. **Build better detectors** -- know which features the model relies on
2. **Understand model failures** -- see when the model misses critical tokens
3. **Craft adversarial attacks** -- manipulate text to redirect attention away from malicious indicators

### Our Test Cases

We compare matched pairs of text from different security domains:

| Category | Benign Example | Malicious Example |
|----------|---------------|-------------------|
| **Email** | Routine team communication | Social engineering / phishing |
| **Log Entry** | Normal authentication event | Brute-force attack pattern |

By using matched pairs (same domain, similar length), we isolate the effect of
malicious content on attention patterns rather than confounding it with topic or length differences.

### What to Look For

Compare the heatmaps side by side:
- Do the same heads attend to the same structural positions?
- Does the malicious text cause attention to concentrate on specific trigger words?
- Are there heads that look nearly identical between benign and malicious (structural heads)
  vs. heads that look very different (semantic heads)?


In [ ]:
# ============================================================
# Cell 6: Prepare Security-Relevant Text Pairs
# ============================================================
# Define benign vs. malicious/suspicious text pairs and
# build a reusable function to extract attention from any text.
# ============================================================

def extract_attention(text, tokenizer, model):
    """
    Extract attention weights and tokens from a text input.

    Returns:
        tokens: list of token strings
        attentions: tuple of attention tensors (one per layer)
    """
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())

    with torch.no_grad():
        outputs = model(**inputs)

    return tokens, outputs.attentions


# ---- Define our security text pairs ----
security_pairs = {
    "Email": {
        "benign": "Hi team, the quarterly report is attached for your review",
        "malicious": "URGENT: Your account has been compromised click here immediately to verify"
    },
    "Log Entry": {
        "benign": "User admin logged in from 192.168.1.1",
        "malicious": "User root failed login attempt from 45.33.32.156 after 50 retries"
    }
}

# Extract attention for all texts
attention_data = {}
for category, pair in security_pairs.items():
    attention_data[category] = {}
    for label, text in pair.items():
        tokens, attentions = extract_attention(text, tokenizer, model)
        attention_data[category][label] = {
            'text': text,
            'tokens': tokens,
            'attentions': attentions
        }
        print(f"[{category}] {label}: {len(tokens)} tokens")
        print(f"  Text: \"{text}\"")
        print(f"  Tokens: {tokens}\n")

print("Attention extracted for all security text pairs.")

### Email Comparison: Benign vs. Phishing

Now we put the comparison into action. The cell below creates **side-by-side attention
heatmaps** for a benign email snippet and a phishing email snippet.

**Benign:** `"Hi team, please find the quarterly report attached for your review"`
**Phishing:** `"URGENT: Your account has been compromised click here immediately to verify your identity"`

We look at Layer 5 (the deepest layer, where attention is most semantic).
The comparison function shows the same head across both texts so you can
directly compare where the model's attention falls.

**Key question:** Does the phishing email cause attention to cluster around
urgency words ("URGENT", "immediately") and action words ("click", "verify")?


In [ ]:
# ============================================================
# Cell 7: Side-by-Side Attention Comparison (Emails)
# ============================================================
# Compare attention patterns between a benign email and a
# phishing email. We look at a specific layer and head to
# see if the model treats them differently.
# ============================================================

def plot_attention_comparison(data_benign, data_malicious, category,
                              layer=0, head=0, cmap="YlOrRd"):
    """
    Plot side-by-side attention heatmaps for benign vs. malicious text.
    """
    tokens_b = data_benign['tokens']
    tokens_m = data_malicious['tokens']
    attn_b = data_benign['attentions'][layer][0, head].detach().numpy()
    attn_m = data_malicious['attentions'][layer][0, head].detach().numpy()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

    # ---- Benign ----
    im1 = ax1.imshow(attn_b, cmap=cmap, aspect='auto', vmin=0, vmax=1)
    ax1.set_xticks(range(len(tokens_b)))
    ax1.set_yticks(range(len(tokens_b)))
    ax1.set_xticklabels(tokens_b, rotation=45, ha='right', fontsize=8)
    ax1.set_yticklabels(tokens_b, fontsize=8)
    ax1.set_title(f'BENIGN {category}\nLayer {layer}, Head {head}',
                  fontsize=13, fontweight='bold', color='green')
    ax1.set_xlabel('Attended To', fontsize=10)
    ax1.set_ylabel('Attending From', fontsize=10)
    plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

    # Add annotations for benign
    ann_fontsize_b = 6 if len(tokens_b) <= 16 else 5 if len(tokens_b) <= 20 else 4
    if len(tokens_b) <= 25:
        for i in range(len(tokens_b)):
            for j in range(len(tokens_b)):
                val = attn_b[i, j]
                color = 'white' if val > 0.5 else 'black'
                ax1.text(j, i, f'{val:.2f}', ha='center', va='center',
                         fontsize=ann_fontsize_b, color=color)

    # ---- Malicious ----
    im2 = ax2.imshow(attn_m, cmap=cmap, aspect='auto', vmin=0, vmax=1)
    ax2.set_xticks(range(len(tokens_m)))
    ax2.set_yticks(range(len(tokens_m)))
    ax2.set_xticklabels(tokens_m, rotation=45, ha='right', fontsize=8)
    ax2.set_yticklabels(tokens_m, fontsize=8)
    ax2.set_title(f'MALICIOUS/SUSPICIOUS {category}\nLayer {layer}, Head {head}',
                  fontsize=13, fontweight='bold', color='red')
    ax2.set_xlabel('Attended To', fontsize=10)
    ax2.set_ylabel('Attending From', fontsize=10)
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

    # Add annotations for malicious
    ann_fontsize_m = 6 if len(tokens_m) <= 16 else 5 if len(tokens_m) <= 20 else 4
    if len(tokens_m) <= 25:
        for i in range(len(tokens_m)):
            for j in range(len(tokens_m)):
                val = attn_m[i, j]
                color = 'white' if val > 0.5 else 'black'
                ax2.text(j, i, f'{val:.2f}', ha='center', va='center',
                         fontsize=ann_fontsize_m, color=color)

    plt.suptitle(f'{category} Comparison: Benign vs. Malicious',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


# Compare emails at Layer 0 and Layer 5
print("EMAIL COMPARISON: Benign vs. Phishing")
print("=" * 60)
print(f"Benign:    \"{security_pairs['Email']['benign']}\"")
print(f"Phishing:  \"{security_pairs['Email']['malicious']}\"")
print()

for layer_idx in [0, num_layers - 1]:
    print(f"\n--- Layer {layer_idx}, Head 0 ---")
    plot_attention_comparison(
        attention_data['Email']['benign'],
        attention_data['Email']['malicious'],
        category='Email',
        layer=layer_idx,
        head=0
    )

### Log Entry Comparison: Normal vs. Brute-Force

Same technique, different domain. Now we compare attention on **log entries**:

**Normal:** `"User admin logged in successfully from 10.0.0.1"`
**Suspicious:** `"Failed login attempt for admin from 192.168.1.100 after 50 failed attempts"`

Log analysis is one of the most common applications of NLP in security operations.
SOC teams process millions of log lines daily, and ML models help triage them.
Understanding what the model attends to in a suspicious log entry reveals whether
it's picking up on the right signals (repeated failures, high attempt counts)
or superficial patterns (just the word "Failed").


In [ ]:
# ============================================================
# Cell 8: Side-by-Side Attention Comparison (Log Entries)
# ============================================================
# Same comparison for log entries: normal login vs. brute-force.
# ============================================================

print("LOG ENTRY COMPARISON: Normal vs. Suspicious")
print("=" * 60)
print(f"Normal:     \"{security_pairs['Log Entry']['benign']}\"")
print(f"Suspicious: \"{security_pairs['Log Entry']['malicious']}\"")
print()

for layer_idx in [0, num_layers - 1]:
    print(f"\n--- Layer {layer_idx}, Head 0 ---")
    plot_attention_comparison(
        attention_data['Log Entry']['benign'],
        attention_data['Log Entry']['malicious'],
        category='Log Entry',
        layer=layer_idx,
        head=0
    )

---
## Part 5: Attention Head Explorer

DistilBERT has **6 layers x 12 heads = 72 individual attention patterns**. So far,
we've only looked at a few. Each head may learn to track different linguistic or
semantic features:

| Head Type | What It Tracks | Example |
|-----------|---------------|---------|
| **Positional** | Nearby tokens (local context) | Head that always attends to the previous/next token |
| **Syntactic** | Grammatical relationships | Subject attending to its verb |
| **Semantic** | Meaning relationships | "URGENT" attending to "verify" |
| **Separator** | Special tokens | Most attention flows to [CLS] or [SEP] |
| **Identity** | Self-attention | Strong diagonal pattern |

### Interactive Exploration

The function below lets you visualize any text + layer + head combination.
Try different combinations to discover what each head specializes in. Some
starting points:

- **Layer 0, Head 6** -- often shows local/positional patterns
- **Layer 3, Head 0** -- often shows syntactic patterns
- **Layer 5, Head 11** -- often shows broad semantic patterns

These are guidelines, not rules. Part of the exercise is discovering which
heads are most interesting for your specific text.


In [ ]:
# ============================================================
# Cell 9: Attention Head Explorer
# ============================================================
# A reusable function that takes any text + layer + head and
# produces a visualization. Students can experiment freely.
# ============================================================

def explore_attention(text, layer, head, cmap="YlOrRd", figsize=(10, 8)):
    """
    All-in-one function: tokenize, extract attention, and visualize
    for any text at any layer+head combination.

    Args:
        text (str): Input text to analyze
        layer (int): Layer index (0 to 5 for DistilBERT)
        head (int): Head index (0 to 11 for DistilBERT)
        cmap (str): Colormap name
        figsize (tuple): Figure size

    Returns:
        tokens: list of tokens
        attn_matrix: numpy array of attention weights
    """
    # Validate inputs
    assert 0 <= layer < num_layers, f"Layer must be 0-{num_layers-1}, got {layer}"
    assert 0 <= head < num_heads, f"Head must be 0-{num_heads-1}, got {head}"

    # Tokenize and run inference
    tokens, attentions = extract_attention(text, tokenizer, model)

    # Visualize
    attn_matrix = visualize_attention_head(
        attentions, tokens, layer, head,
        title_suffix=f'\n\"{text[:60]}...\"' if len(text) > 60 else f'\n\"{text}\"',
        figsize=figsize, cmap=cmap
    )

    return tokens, attn_matrix



# ============================================================
# >>> STUDENT: MODIFY THESE VALUES <<<
# ============================================================
# Change demo_text to analyze any text you want.
# Change the (layer, head) tuples to explore different heads.
#
# layer: 0-5  (0 = shallowest, 5 = deepest)
# head:  0-11 (each head learns different patterns)
# ============================================================

# CHANGE THIS: Try your own security-relevant text
demo_text = "URGENT: Your account has been compromised click here immediately to verify"

# CHANGE THESE: Pick any (layer, head) combinations to visualize
interesting_heads = [
    (0, 0),    # Layer 0, Head 0  — typically local/positional patterns
    (0, 6),    # Layer 0, Head 6  — often attends to adjacent tokens
    (2, 3),    # Layer 2, Head 3  — mid-depth, syntactic patterns
    (5, 0),    # Layer 5, Head 0  — deepest layer, semantic patterns
    (5, 11),   # Layer 5, Head 11 — deepest layer, another perspective
]

# ============================================================
# No changes needed below this line
# ============================================================

print("ATTENTION HEAD EXPLORER")
print("=" * 60)
print(f"Analyzing: \"{demo_text}\"")
print(f"Exploring {len(interesting_heads)} layer+head combinations:\n")

for layer_idx, head_idx in interesting_heads:
    print(f"\n>>> Layer {layer_idx}, Head {head_idx}")
    _, _ = explore_attention(demo_text, layer=layer_idx, head=head_idx)

---
## Part 6: Average Attention Patterns Across Layers

Individual heads capture specific patterns, but the **average** across all
heads in a layer shows the overall attention strategy at that depth.

### How Attention Evolves Through Layers

Research (Clark et al., 2019) has shown a consistent progression:

| Layer Depth | Typical Attention Pattern | Interpretation |
|-------------|--------------------------|----------------|
| **Early (0-1)** | Strong local attention (nearby tokens) | Building word-level representations |
| **Middle (2-3)** | Mix of local and long-range | Syntactic relationships forming |
| **Late (4-5)** | Broad, distributed attention | Semantic understanding, topic modeling |

### Why This Matters for Security

This layer-by-layer progression means:
- **Early layers** might catch surface-level phishing signals (suspicious characters, unusual formatting)
- **Late layers** catch semantic signals (urgency + action + credentials = phishing intent)
- A model that only uses early-layer features might be easier to evade with obfuscation

### What We Compute

We average the attention across all 12 heads for each of the 6 layers,
producing 6 heatmaps that show how the model's "focus strategy" evolves
from shallow pattern matching to deep semantic understanding.


In [ ]:
# ============================================================
# Cell 10: Average Attention Patterns Per Layer
# ============================================================
# Average attention across all 12 heads for each layer.
# Shows how the model's attention strategy evolves from
# shallow to deep layers.
# ============================================================

text = "URGENT: Your account has been compromised click here immediately to verify"
tokens, attentions = extract_attention(text, tokenizer, model)

fig, axes = plt.subplots(2, 3, figsize=(24, 14))
fig.suptitle('Average Attention Per Layer (Averaged Across All 12 Heads)',
             fontsize=16, fontweight='bold', y=1.02)

for layer_idx in range(num_layers):
    ax = axes[layer_idx // 3, layer_idx % 3]

    # Average across all heads: shape goes from (1, 12, seq, seq) to (seq, seq)
    avg_attn = attentions[layer_idx][0].mean(dim=0).detach().numpy()

    im = ax.imshow(avg_attn, cmap='YlOrRd', aspect='auto', vmin=0)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(tokens, fontsize=7)
    ax.set_title(f'Layer {layer_idx} (avg over 12 heads)', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# ---- Quantify attention spread per layer ----
print("\nATTENTION ENTROPY PER LAYER")
print("(Higher entropy = more distributed/broader attention)")
print("=" * 50)
for layer_idx in range(num_layers):
    avg_attn = attentions[layer_idx][0].mean(dim=0).detach().numpy()
    # Compute average entropy across all tokens
    # Entropy = -sum(p * log(p))
    entropy = -np.sum(avg_attn * np.log(avg_attn + 1e-10), axis=-1).mean()
    max_entropy = np.log(len(tokens))  # Maximum possible entropy (uniform)
    normalized = entropy / max_entropy
    bar = '#' * int(normalized * 40)
    print(f"  Layer {layer_idx}: entropy={entropy:.3f} "
          f"(normalized={normalized:.2%}) |{bar}|")

---
## Part 7: [CLS] Token Analysis -- The Classification Signal

In BERT-style models, the **[CLS] token** is special. It is prepended to
every input and its final hidden state is used as the **aggregate sequence
representation** for classification tasks.

### Why [CLS] Attention Matters

What the [CLS] token attends to effectively tells us **what the model considers
most important** about the entire input. This is because:

1. [CLS] has no inherent meaning -- it learns to be a "summary" token
2. Through training, it learns to attend to the tokens most relevant for the task
3. Its final hidden state is what gets fed to the classification head

For security classification (e.g., phishing detection, malware classification):
- If [CLS] attends heavily to "URGENT" and "verify" in a phishing email, the model
  has learned these are strong phishing indicators
- If [CLS] distributes attention evenly, the model may not have strong signal

### What We Visualize

We extract the [CLS] token's attention distribution from the **last layer** (where
it's most refined) and plot it as a bar chart. Each bar shows how much attention
[CLS] pays to each token in the input. We compare benign vs. malicious text
side by side to see if [CLS] "looks at" different things.


In [ ]:
# ============================================================
# Cell 11: [CLS] Token Attention Analysis
# ============================================================
# The [CLS] token's attention pattern shows what the model
# considers most important for classification. We compare
# [CLS] attention between benign and malicious inputs.
# ============================================================

def plot_cls_attention(text, label, color, ax, layer=-1):
    """
    Plot what the [CLS] token attends to (averaged across heads)
    at a given layer. Uses the last layer by default.
    """
    tokens, attentions = extract_attention(text, tokenizer, model)

    if layer == -1:
        layer = num_layers - 1

    # [CLS] is always token index 0
    # Average across all heads, get row 0 (CLS attending to all tokens)
    cls_attn = attentions[layer][0].mean(dim=0)[0].detach().numpy()

    # Plot as bar chart
    bars = ax.bar(range(len(tokens)), cls_attn, color=color, alpha=0.8,
                  edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Attention Weight', fontsize=10)
    ax.set_title(f'[CLS] Attention ({label})\nLayer {layer}, avg across heads',
                 fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(cls_attn) * 1.2)

    # Highlight top-3 attended tokens
    top_indices = np.argsort(cls_attn)[-3:]
    for idx in top_indices:
        bars[idx].set_edgecolor('red')
        bars[idx].set_linewidth(2.5)
        ax.text(idx, cls_attn[idx] + 0.005, tokens[idx],
                ha='center', va='bottom', fontsize=8, fontweight='bold', color='red')

    return tokens, cls_attn


# ---- Compare CLS attention: Benign vs. Phishing Email ----
print("[CLS] TOKEN ATTENTION ANALYSIS")
print("=" * 60)
print("What does the [CLS] token (the classification signal) attend to?")
print("Red-outlined bars = top-3 most attended tokens\n")

fig, axes = plt.subplots(2, 2, figsize=(20, 12))

# Emails
tokens_b, cls_b = plot_cls_attention(
    security_pairs['Email']['benign'], 'Benign Email', '#2ecc71', axes[0, 0])
tokens_m, cls_m = plot_cls_attention(
    security_pairs['Email']['malicious'], 'Phishing Email', '#e74c3c', axes[0, 1])

# Log entries
tokens_lb, cls_lb = plot_cls_attention(
    security_pairs['Log Entry']['benign'], 'Normal Log', '#2ecc71', axes[1, 0])
tokens_lm, cls_lm = plot_cls_attention(
    security_pairs['Log Entry']['malicious'], 'Suspicious Log', '#e74c3c', axes[1, 1])

plt.suptitle('[CLS] Token Attention: What the Model Considers Important for Classification',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print the top attended tokens
print("\nTOP-3 TOKENS BY [CLS] ATTENTION:")
print("-" * 50)
for name, tokens, cls_attn in [
    ("Benign Email", tokens_b, cls_b),
    ("Phishing Email", tokens_m, cls_m),
    ("Normal Log", tokens_lb, cls_lb),
    ("Suspicious Log", tokens_lm, cls_lm)
]:
    top3 = np.argsort(cls_attn)[-3:][::-1]
    top_tokens = [(tokens[i], f"{cls_attn[i]:.4f}") for i in top3]
    print(f"  {name:20s}: {top_tokens}")

---
## Part 8: Attention Difference Analysis

Visual comparison is good, but **quantitative analysis** is better. In this section,
we compute statistical differences in attention patterns between benign and malicious text.

### What We Measure

| Metric | What It Tells Us |
|--------|-----------------|
| **Entropy** | How spread out the attention is. High entropy = distributed attention. Low entropy = focused on a few tokens. |
| **Max attention** | The highest single attention weight. High max = one token dominates. |
| **Self-attention** | How much tokens attend to themselves (diagonal of the matrix). |
| **[CLS] concentration** | How focused [CLS] attention is (measured by top-3 attention share). |

### Why This Matters

If malicious text consistently produces **lower entropy** (more focused attention) than
benign text, it means the model is "locking on" to specific tokens -- possibly the same
tokens a security analyst would flag. This gives us a quantitative signal that could be
used as an additional feature in a detection pipeline.

### What to Look For

Compare the statistics between benign and malicious text. Key questions:
- Does malicious text produce more focused attention (lower entropy)?
- Do different layers show different statistical signatures?
- Which metrics show the biggest difference between benign and malicious?


In [ ]:
# ============================================================
# Cell 12: Attention Statistics Comparison
# ============================================================
# Compute quantitative metrics comparing attention patterns
# between benign and malicious inputs.
# ============================================================

def compute_attention_stats(text, tokenizer, model):
    """
    Compute summary statistics of attention patterns for a given text.

    Returns a dict with per-layer stats:
    - max_attention: highest single attention weight
    - avg_entropy: average entropy across tokens
    - diagonal_strength: how much tokens attend to themselves
    - cls_concentration: how concentrated CLS attention is (max / mean)
    """
    tokens, attentions = extract_attention(text, tokenizer, model)
    seq_len = len(tokens)
    stats = {}

    for layer_idx in range(len(attentions)):
        # Average across heads
        avg_attn = attentions[layer_idx][0].mean(dim=0).detach().numpy()

        # Max attention weight in the matrix
        max_attn = avg_attn.max()

        # Average entropy across tokens
        entropy = -np.sum(avg_attn * np.log(avg_attn + 1e-10), axis=-1).mean()

        # Diagonal strength (self-attention)
        diag = np.diag(avg_attn).mean()

        # CLS concentration
        cls_row = avg_attn[0]
        cls_concentration = cls_row.max() / (cls_row.mean() + 1e-10)

        stats[f"layer_{layer_idx}"] = {
            'max_attention': float(max_attn),
            'avg_entropy': float(entropy),
            'diagonal_strength': float(diag),
            'cls_concentration': float(cls_concentration)
        }

    return stats


# Compute stats for all our security texts
print("ATTENTION STATISTICS: BENIGN vs. MALICIOUS")
print("=" * 70)

for category, pair in security_pairs.items():
    print(f"\n{'='*70}")
    print(f"  {category.upper()}")
    print(f"{'='*70}")

    stats_benign = compute_attention_stats(pair['benign'], tokenizer, model)
    stats_malicious = compute_attention_stats(pair['malicious'], tokenizer, model)

    print(f"\n  {'Metric':<25s} {'Layer':<8s} {'Benign':>10s} {'Malicious':>10s} {'Delta':>10s}")
    print(f"  {'-'*63}")

    for layer_key in stats_benign:
        layer_num = layer_key.split('_')[1]
        for metric in ['max_attention', 'avg_entropy', 'diagonal_strength', 'cls_concentration']:
            b_val = stats_benign[layer_key][metric]
            m_val = stats_malicious[layer_key][metric]
            delta = m_val - b_val
            marker = ' **' if abs(delta) > 0.1 else ''
            print(f"  {metric:<25s} {layer_num:<8s} {b_val:>10.4f} {m_val:>10.4f} {delta:>+10.4f}{marker}")
        print()

print("\n** = Notable difference (delta > 0.1)")
print("\nKey observations:")
print("- 'cls_concentration': How focused the [CLS] token's attention is")
print("  Higher values = model focuses on fewer tokens for classification")
print("- 'diagonal_strength': How much tokens attend to themselves")
print("- 'avg_entropy': Higher = more distributed attention")

---
## Part 9: Multi-Head Attention Grid

Let's see **ALL 12 heads at once** for a single layer. This is the most complete
view of what a single transformer layer is doing.

### Why View All Heads Together?

Each attention head operates independently within a layer. Viewing them all at
once reveals the **diversity of representations** the model maintains:

- Some heads will show nearly identical patterns (redundancy / robustness)
- Some heads will show completely unique patterns (specialization)
- The combination of all 12 heads is what gives the layer its expressive power

### Head Specialization Examples

In BERT-family models, researchers have found heads that specialize in:
- **Direct object detection** -- verb tokens attend to their objects
- **Coreference** -- pronouns attend to their referents
- **Positional** -- fixed-offset attention (always attend to token at position i-1)
- **Separator focusing** -- heavy attention to [SEP] tokens

### What to Look For

In the 4x3 grid of heatmaps:
- Which heads look similar to each other?
- Which heads stand out as unique?
- Can you identify any of the specialization patterns described above?


In [ ]:
# ============================================================
# Cell 13: Multi-Head Attention Grid
# ============================================================
# Show all 12 attention heads for a given layer in a grid.
# This reveals how different heads specialize.
# ============================================================

def plot_all_heads(text, layer, figsize=(28, 20)):
    """
    Plot all attention heads for a given layer in a 3x4 grid.
    """
    tokens, attentions = extract_attention(text, tokenizer, model)

    fig, axes = plt.subplots(3, 4, figsize=figsize)
    fig.suptitle(f'All {num_heads} Attention Heads — Layer {layer}\n"{text}"',
                 fontsize=16, fontweight='bold', y=1.02)

    for head_idx in range(num_heads):
        row, col = head_idx // 4, head_idx % 4
        ax = axes[row, col]

        attn = attentions[layer][0, head_idx].detach().numpy()
        im = ax.imshow(attn, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

        ax.set_xticks(range(len(tokens)))
        ax.set_yticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=6)
        ax.set_yticklabels(tokens, fontsize=6)
        ax.set_title(f'Head {head_idx}', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.show()


# Show all heads for the phishing email at Layer 0 and the last layer
phishing_text = "URGENT: Your account has been compromised click here immediately to verify"

print("ALL HEADS AT LAYER 0 (initial attention patterns):")
plot_all_heads(phishing_text, layer=0)

print(f"\nALL HEADS AT LAYER {num_layers-1} (final/deepest attention patterns):")
plot_all_heads(phishing_text, layer=num_layers-1)

---
## Part 10: Token Importance via Attention Rollout

Individual layer attention only tells part of the story. When we look at Layer 3,
we're seeing attention at that specific depth -- but information flows through
**ALL layers** via residual connections.

### The Problem with Single-Layer Attention

If Token A attends to Token B in Layer 2, and Token B attends to Token C in Layer 3,
then Token A is **indirectly attending to Token C** even though no single layer's
attention matrix shows this connection. Single-layer heatmaps miss these transitive flows.

### Attention Rollout (Abnar & Zuidema, 2020)

Attention rollout solves this by **multiplying attention matrices across layers**:

```
R(0) = A(0)                    # First layer attention
R(1) = A(1) × R(0)            # Layer 1 attention × previous flow
R(2) = A(2) × R(1)            # Layer 2 attention × cumulative flow
...
R(L) = A(L) × R(L-1)          # Final rollout matrix
```

At each step, the identity matrix is mixed in (via residual connections),
ensuring that each token retains some attention to itself.

### What Rollout Reveals

The final rollout matrix shows the **total information flow** from each input
token to the [CLS] representation. This is the most complete picture of
"what the model actually uses" for its final output.

### What to Look For

We plot rollout-based token importance for both benign and malicious text.
Compare which tokens accumulate the most total attention flow. Do the
high-importance tokens align with what a security analyst would flag?


In [ ]:
# ============================================================
# Cell 14: Attention Rollout
# ============================================================
# Compute attention rollout to trace information flow across
# ALL layers, not just one. This accounts for residual
# connections by mixing attention with identity matrices.
# ============================================================

def attention_rollout(attentions, head_fusion='mean'):
    """
    Compute attention rollout across all layers.

    Attention rollout accounts for the residual connections in the
    transformer by computing:
      rollout = product of (0.5 * I + 0.5 * A_l) for all layers l

    This gives a more complete picture of how information flows
    from input tokens to the final representation.

    Args:
        attentions: tuple of attention tensors from model
        head_fusion: how to combine heads ('mean', 'max', 'min')

    Returns:
        rollout_matrix: (seq_len, seq_len) numpy array
    """
    # Start with identity matrix
    result = None

    for layer_attn in attentions:
        # Fuse heads
        if head_fusion == 'mean':
            attn_heads_fused = layer_attn[0].mean(dim=0)
        elif head_fusion == 'max':
            attn_heads_fused = layer_attn[0].max(dim=0).values
        elif head_fusion == 'min':
            attn_heads_fused = layer_attn[0].min(dim=0).values

        # Add residual connection (identity matrix)
        seq_len = attn_heads_fused.shape[0]
        I = torch.eye(seq_len)
        attn_with_residual = 0.5 * attn_heads_fused + 0.5 * I

        # Re-normalize rows to sum to 1
        attn_with_residual = attn_with_residual / attn_with_residual.sum(dim=-1, keepdim=True)

        if result is None:
            result = attn_with_residual
        else:
            result = torch.matmul(attn_with_residual, result)

    return result.detach().numpy()


def plot_rollout_comparison(texts, labels, colors):
    """
    Compare attention rollout for the [CLS] token across multiple texts.
    Shows which input tokens contribute most to the final [CLS] representation.
    """
    n = len(texts)
    fig, axes = plt.subplots(1, n, figsize=(8 * n, 5))
    if n == 1:
        axes = [axes]

    for idx, (text, label, color) in enumerate(zip(texts, labels, colors)):
        tokens, attentions = extract_attention(text, tokenizer, model)
        rollout = attention_rollout(attentions)

        # CLS token's rollout attention (row 0)
        cls_rollout = rollout[0]

        ax = axes[idx]
        bars = ax.bar(range(len(tokens)), cls_rollout, color=color, alpha=0.8,
                      edgecolor='black', linewidth=0.5)
        ax.set_xticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Rollout Attention', fontsize=10)
        ax.set_title(f'{label}', fontsize=12, fontweight='bold')

        # Highlight top-3
        top3 = np.argsort(cls_rollout)[-3:]
        for i in top3:
            bars[i].set_edgecolor('red')
            bars[i].set_linewidth(2.5)
            ax.text(i, cls_rollout[i] + 0.002, tokens[i],
                    ha='center', va='bottom', fontsize=8, fontweight='bold', color='red')

    plt.suptitle('[CLS] Attention Rollout: Total Information Flow Across All Layers',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


# Compare benign vs. phishing with rollout
print("ATTENTION ROLLOUT: BENIGN vs. PHISHING EMAIL")
print("=" * 60)
print("Unlike single-layer attention, rollout traces information")
print("flow through ALL layers, accounting for residual connections.\n")

plot_rollout_comparison(
    texts=[
        security_pairs['Email']['benign'],
        security_pairs['Email']['malicious']
    ],
    labels=['Benign Email', 'Phishing Email'],
    colors=['#2ecc71', '#e74c3c']
)

print("\nATTENTION ROLLOUT: NORMAL vs. SUSPICIOUS LOG")
print("=" * 60)

plot_rollout_comparison(
    texts=[
        security_pairs['Log Entry']['benign'],
        security_pairs['Log Entry']['malicious']
    ],
    labels=['Normal Log Entry', 'Suspicious Log Entry'],
    colors=['#2ecc71', '#e74c3c']
)

---
## Student Exercises

Now it is your turn. Use the functions we have built to explore attention  
patterns on your own. Each exercise builds on the previous one.

### Exercise 1: Input Your Own Text
Use the `explore_attention()` function to visualize attention on  
your own security-relevant text. Try:
- A SQL injection payload
- A benign SQL query
- A suspicious command-line entry
- A normal system log message

### Exercise 2: Find the Most Discriminative Head
Which layer+head combination best distinguishes phishing from  
legitimate email? Hint: loop over all 72 heads and compare the  
[CLS] attention distributions using a distance metric.

### Exercise 3: Top Attended Tokens
For the suspicious content, which tokens consistently receive  
the most attention? Do urgency words ("URGENT", "immediately")  
or action words ("click", "verify") get more attention?

In [ ]:
# ============================================================
# Exercise 1: Input Your Own Text
# ============================================================
# Modify the text below and run this cell to see its attention.
# Try different layer and head combinations!
# ============================================================

# ---- MODIFY THESE ----
your_text = "SELECT * FROM users WHERE username = 'admin' OR 1=1 --"
your_layer = 3   # Try 0 through 5
your_head = 0    # Try 0 through 11
# ----------------------

print(f"Your text: \"{your_text}\"")
print(f"Layer: {your_layer}, Head: {your_head}\n")

tokens, attn_matrix = explore_attention(your_text, layer=your_layer, head=your_head)

In [ ]:
# ============================================================
# Exercise 2: Find the Most Discriminative Head
# ============================================================
# Loop over all 72 heads and find which one shows the biggest
# difference between benign and phishing email attention.
# We compare entropy and concentration of [CLS] attention
# distributions between benign and malicious inputs.
# ============================================================

benign_text = security_pairs['Email']['benign']
phishing_text = security_pairs['Email']['malicious']

tokens_b, attentions_b = extract_attention(benign_text, tokenizer, model)
tokens_p, attentions_p = extract_attention(phishing_text, tokenizer, model)

print("SEARCHING FOR THE MOST DISCRIMINATIVE ATTENTION HEAD")
print("=" * 60)
print(f"Comparing [CLS] attention between:")
print(f"  Benign:   \"{benign_text}\"")
print(f"  Phishing: \"{phishing_text}\"\n")

# Note: Because the two texts have different lengths, we compare
# the entropy and concentration rather than direct distribution distance.
# We compute a composite discriminability score.
results = []

for layer_idx in range(num_layers):
    for head_idx in range(num_heads):
        # Get CLS attention for each
        cls_b = attentions_b[layer_idx][0, head_idx, 0].detach().numpy()
        cls_p = attentions_p[layer_idx][0, head_idx, 0].detach().numpy()

        # Entropy of CLS attention
        entropy_b = -np.sum(cls_b * np.log(cls_b + 1e-10))
        entropy_p = -np.sum(cls_p * np.log(cls_p + 1e-10))

        # Max attention (concentration)
        max_b = cls_b.max()
        max_p = cls_p.max()

        # Discriminability: absolute difference in entropy + max attention
        score = abs(entropy_b - entropy_p) + abs(max_b - max_p)

        results.append({
            'layer': layer_idx,
            'head': head_idx,
            'score': score,
            'entropy_benign': entropy_b,
            'entropy_phishing': entropy_p,
            'max_benign': max_b,
            'max_phishing': max_p
        })

# Sort by discriminability
results.sort(key=lambda x: x['score'], reverse=True)

print(f"{'Rank':<6s} {'Layer':<8s} {'Head':<8s} {'Score':<10s} "
      f"{'Ent(B)':<10s} {'Ent(P)':<10s} {'Max(B)':<10s} {'Max(P)':<10s}")
print("-" * 70)

for rank, r in enumerate(results[:10], 1):
    print(f"{rank:<6d} {r['layer']:<8d} {r['head']:<8d} {r['score']:<10.4f} "
          f"{r['entropy_benign']:<10.4f} {r['entropy_phishing']:<10.4f} "
          f"{r['max_benign']:<10.4f} {r['max_phishing']:<10.4f}")

# Visualize the top discriminative head
best = results[0]
print(f"\nMost discriminative head: Layer {best['layer']}, Head {best['head']}")
print(f"Score: {best['score']:.4f}")
print("\nVisualizing this head on both texts:\n")

print("--- Benign Email ---")
_ = explore_attention(benign_text, layer=best['layer'], head=best['head'])

print("\n--- Phishing Email ---")
_ = explore_attention(phishing_text, layer=best['layer'], head=best['head'])

In [ ]:
# ============================================================
# Exercise 3: Token Importance Ranking
# ============================================================
# For the suspicious/malicious texts, which tokens receive
# the most total attention? We aggregate attention received
# by each token across all layers and heads.
# ============================================================

def rank_token_importance(text, tokenizer, model):
    """
    Rank tokens by how much total attention they RECEIVE
    (column-wise sum across all layers and heads).

    A token that receives high attention from many other tokens
    is a key information hub in the sentence.
    """
    tokens, attentions = extract_attention(text, tokenizer, model)

    # Sum attention received by each token across all layers and heads
    total_received = np.zeros(len(tokens))

    for layer_attn in attentions:
        for head_idx in range(layer_attn.shape[1]):
            attn_matrix = layer_attn[0, head_idx].detach().numpy()
            # Column sum = total attention received by each token
            total_received += attn_matrix.sum(axis=0)

    # Normalize
    total_received = total_received / total_received.sum()

    return tokens, total_received


print("TOKEN IMPORTANCE RANKING (by total attention received)")
print("=" * 60)
print("Which tokens are 'information hubs' that other tokens attend to?\n")

all_texts = [
    ("Benign Email", security_pairs['Email']['benign'], '#2ecc71'),
    ("Phishing Email", security_pairs['Email']['malicious'], '#e74c3c'),
    ("Normal Log", security_pairs['Log Entry']['benign'], '#2ecc71'),
    ("Suspicious Log", security_pairs['Log Entry']['malicious'], '#e74c3c'),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes_flat = axes.flatten()

for idx, (label, text, color) in enumerate(all_texts):
    tokens, importance = rank_token_importance(text, tokenizer, model)

    ax = axes_flat[idx]
    bars = ax.bar(range(len(tokens)), importance, color=color, alpha=0.8,
                  edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Normalized Attention Received', fontsize=9)
    ax.set_title(f'{label}', fontsize=12, fontweight='bold')

    # Highlight top-3 most important (excluding [CLS] and [SEP])
    # Skip index 0 ([CLS]) and last ([SEP]) for more interesting results
    content_importance = importance.copy()
    content_importance[0] = 0   # Exclude [CLS]
    content_importance[-1] = 0  # Exclude [SEP]
    top3 = np.argsort(content_importance)[-3:][::-1]

    for i in top3:
        bars[i].set_edgecolor('red')
        bars[i].set_linewidth(2.5)
        ax.text(i, importance[i] + 0.002, tokens[i],
                ha='center', va='bottom', fontsize=8, fontweight='bold', color='red')

    # Print ranking
    print(f"\n{label}:")
    ranked = sorted(enumerate(zip(tokens, importance)),
                    key=lambda x: x[1][1], reverse=True)
    for rank, (i, (tok, imp)) in enumerate(ranked[:5], 1):
        print(f"  #{rank}: '{tok}' (importance: {imp:.4f})")

plt.suptitle('Token Importance: Total Attention Received Across All Layers & Heads',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Discussion

### What We Observed

1. **Attention is not random** — the model develops structured patterns that
   differ systematically between benign and malicious content.

2. **Different heads capture different features** — some focus on local
   syntax, others on long-range semantic relationships. This diversity is
   what makes multi-head attention powerful.

3. **Deeper layers have broader attention** — early layers focus locally
   while later layers integrate information across the entire sequence.

4. **[CLS] attention reveals classification priorities** — the tokens
   the [CLS] position attends to are effectively what the model uses
   to make classification decisions.

### Connection to Adversarial Attacks

Understanding attention patterns opens the door to **adversarial attacks**  
on NLP-based security classifiers:

- **Attention manipulation:** If you know the model focuses on urgency words
  like "URGENT" for phishing detection, you can craft phishing emails that
  avoid those trigger tokens.

- **Attention flooding:** Adding many benign tokens to dilute the model's
  attention away from the malicious payload.

- **Token-level evasion:** Replacing high-attention tokens with synonyms
  or misspellings that the tokenizer splits differently.

### Connection to Defense

- **Attention-based detection:** Monitor attention concentration — highly
  concentrated attention on unusual tokens may indicate adversarial input.

- **Interpretability for trust:** Attention visualization lets security
  analysts verify that a classifier is detecting threats for the right
  reasons, not relying on spurious correlations.

### Important Limitations

- **Attention is not explanation** — Jain & Wallace (2019) showed that
  attention weights don't always faithfully represent feature importance.
  They are one interpretability signal among many.

- **DistilBERT is not fine-tuned** — we used the base model, not one
  fine-tuned for security classification. A fine-tuned model's attention
  patterns would differ and be more task-specific.

- **Attention rollout is an approximation** — it assumes linear information
  flow and does not fully capture nonlinear layer interactions.

---
## Key Takeaways

1. **Transformers are inspectable** — we can extract and visualize exactly
   what the model attends to, making the "black box" partially transparent.

2. **Attention differs between benign and malicious text** — the model
   processes security-relevant features in measurably different ways.

3. **Multi-head attention provides diversity** — 72 different attention
   patterns (6 layers x 12 heads) capture different aspects of language.

4. **[CLS] attention is the classification signal** — understanding what
   it attends to reveals the model's decision basis.

5. **Interpretability enables both attack and defense** — knowing what the
   model sees helps attackers evade it AND defenders strengthen it.

6. **This is foundational for the rest of the workshop** — understanding
   transformer internals is prerequisite for adversarial ML, prompt
   injection, and the abliteration lab that follows.

---

**Next:** Lab 3 will build on this foundation to explore how adversarial
inputs can manipulate transformer behavior, and how defenders can detect
and mitigate these attacks.

---
*Workshop: Attacking, Defending & Leveraging Agentic AI*  
*Derek Banks and Dr. Brian Fehrman*  
*Lab 2 of 4*